# 🧬 Práctica B: Anotación de Genomas con conda en Google Colab

> **Antes de continuar**, lea la [guía de prácticas compartida](00_genome_annotation_common.md) — contiene el contexto biológico de cada caso, los conceptos clave, las plataformas y la descripción de los archivos de salida.

Esta práctica sigue el mismo flujo de trabajo que la **Práctica A** (Galaxy), pero ejecutando las herramientas desde la línea de comandos en un entorno Google Colab + conda. Esto le permite:

- Instalar y usar herramientas bioinformáticas de CLI sin necesitar un servidor institucional.
- Integrar los resultados directamente en Python con `pandas` y `matplotlib`.
- Comprender cómo se encadenan estas herramientas en un pipeline automatizado.

---

## ⚙️ Paso 0 — Configurar el entorno

Ejecute esta celda **primero**. Instala Miniconda y todas las herramientas necesarias.
⏱️ Puede tardar 8–12 minutos.

In [ ]:
%%bash
# Instalar Miniconda
wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
bash miniconda.sh -b -p /opt/conda
export PATH="/opt/conda/bin:$PATH"

# Instalar herramientas bioinformáticas
conda install -y -c bioconda -c conda-forge \
    bakta \
    ncbi-amrfinderplus \
    plasmidfinder \
    integronfinder \
    isescan \
    antismash

echo "✅ Instalación completa"

## 📥 Paso 1 — Descargar los datos del caso asignado

Descomente **únicamente** el bloque del caso que le indicó el profesor.
Consulte la [guía compartida](00_genome_annotation_common.md#-casos-de-estudio) para el contexto biológico de cada caso.

In [ ]:
%%bash
export PATH="/opt/conda/bin:$PATH"

# ── CASO A: Staphylococcus aureus MRSA (muestra KUN1163) ────────────────────
mkdir -p annotation/caso_A && cd annotation/caso_A
wget -q https://zenodo.org/records/17252812/files/DRR187559_contigs.fasta \
     -O contigs.fasta
echo "✅ Caso A listo — $(grep -c '>' contigs.fasta) contigs"

# ── CASO B: Klebsiella pneumoniae (muestra G20000754) ── (descomente si aplica)
# mkdir -p annotation/caso_B && cd annotation/caso_B
# wget -q https://zenodo.org/records/17252812/files/ERR14828471_contigs.fasta \
#      -O contigs.fasta
# echo "✅ Caso B listo — $(grep -c '>' contigs.fasta) contigs"

# ── CASO C: Streptomyces venezuelae ATCC 10712 ─────── (descomente si aplica)
# mkdir -p annotation/caso_C && cd annotation/caso_C
# wget -q "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/253/235/GCF_000253235.1_ASM25323v1/GCF_000253235.1_ASM25323v1_genomic.fna.gz" \
#      -O GCF_000253235.1_genomic.fna.gz
# gunzip GCF_000253235.1_genomic.fna.gz
# mv GCF_000253235.1_genomic.fna contigs.fasta   # renombramos para uniformidad
# echo "✅ Caso C listo — $(grep -c '>' contigs.fasta) secuencias"

## 🏷️ Paso 2 — Anotación principal con Bakta

Bakta realiza la anotación estructural (CDS, ARNt, ARNr, ncRNA, CRISPR) y funcional en un solo paso.

> **Nota:** Bakta requiere su base de datos. La descargamos en formato light (~1 GB) — suficiente para esta práctica.

In [ ]:
%%bash
export PATH="/opt/conda/bin:$PATH"

# Descargar base de datos de Bakta (versión light)
bakta_db download --output /opt/bakta_db --type light
echo "✅ Base de datos Bakta lista"

In [ ]:
%%bash
export PATH="/opt/conda/bin:$PATH"
CASO="caso_A"   # ← cambie a caso_B si corresponde

bakta \
  --db /opt/bakta_db/db-light \
  --output annotation/${CASO}/bakta_results \
  --prefix genome \
  --keep-contig-headers \
  annotation/${CASO}/contigs.fasta

echo "✅ Bakta completo. Resultados en annotation/${CASO}/bakta_results/"

## 📊 Paso 3 — Explorar el resumen de Bakta con Python

In [ ]:
import pathlib, re

CASO = "caso_A"  # ← cambie a caso_B si corresponde
txt_path = pathlib.Path(f"annotation/{CASO}/bakta_results/genome.txt")

print(txt_path.read_text())

In [ ]:
# Extraer métricas clave del resumen
import pandas as pd

txt = txt_path.read_text()
metricas = {}
for linea in txt.splitlines():
    if ":" in linea:
        clave, _, valor = linea.partition(":")
        metricas[clave.strip()] = valor.strip()

df_resumen = pd.DataFrame.from_dict(metricas, orient="index", columns=["Valor"])
print(df_resumen.to_string())

**Preguntas:**

1. ¿Cuántos contigs había en el input?
2. ¿Cuál es la longitud total del borrador del genoma (*draft genome length*)?
3. ¿Cuántos CDS fueron predichos?
4. ¿Cuántas *small proteins* se encontraron?
5. ¿Cuántos ARNt y ARNr se anotaron?
6. ¿Hay secuencias CRISPR?
7. Compare sus resultados con la **Tabla 1 de Hikichi et al. 2019** (Caso A) o el artículo de referencia (Caso B). ¿Coinciden?

## 🦠 Paso 4 — Genes de resistencia con AMRFinderPlus

In [ ]:
%%bash
export PATH="/opt/conda/bin:$PATH"
CASO="caso_A"           # ← cambie si aplica
ORGANISMO="Staphylococcus_aureus"
# Caso B → ORGANISMO="Klebsiella_pneumoniae"
# Caso C → comente la línea --organism (no existe grupo taxonómico para Streptomyces)

# Actualizar base de datos AMRFinder
amrfinder -u

mkdir -p annotation/${CASO}/amrfinder_results

amrfinder \
  --nucleotide annotation/${CASO}/contigs.fasta \
  --organism ${ORGANISMO} \
  --output annotation/${CASO}/amrfinder_results/amrfinder.tsv \
  --threads 4

echo "✅ AMRFinderPlus completo"

In [ ]:
import pandas as pd

CASO = "caso_A"
amr_df = pd.read_csv(f"annotation/{CASO}/amrfinder_results/amrfinder.tsv", sep="\t")
print(f"Total de genes encontrados: {len(amr_df)}\n")
print(amr_df[["Gene symbol", "Sequence name", "Class", "Subclass", "% Identity to reference sequence"]].to_string())

**Preguntas:**

8. ¿Qué genes de resistencia a antibióticos se encontraron?
9. ¿A qué familias de antibióticos pertenecen (columna `Class`)?
10. ¿En qué contigs se encuentran?
11. ¿Se encontraron factores de virulencia?

## 🔵 Paso 5 — Identificación de plásmidos con PlasmidFinder

In [ ]:
%%bash
export PATH="/opt/conda/bin:$PATH"
CASO="caso_A"

mkdir -p annotation/${CASO}/plasmidfinder_results

plasmidfinder.py \
  -i annotation/${CASO}/contigs.fasta \
  -o annotation/${CASO}/plasmidfinder_results \
  -p /opt/conda/share/plasmidfinder/database \
  -x

echo "✅ PlasmidFinder completo"

In [ ]:
import pandas as pd

CASO = "caso_A"
pf_path = f"annotation/{CASO}/plasmidfinder_results/results_tab.tsv"
pf_df = pd.read_csv(pf_path, sep="\t")
print(f"Plásmidos encontrados: {len(pf_df)}\n")
print(pf_df[["Plasmid", "Identity", "Query / Template length", "Contig", "Position in contig"]].to_string())

**Preguntas:**

12. ¿Cuántos plásmidos se identificaron?
13. ¿A qué replicones / familias pertenecen?
14. ¿En qué contigs se encuentran?
15. Cruzando con AMRFinderPlus: ¿algún gen de resistencia está en el mismo contig que un plásmido?

## 🔗 Paso 6 — Detección de integrones con IntegronFinder

In [ ]:
%%bash
export PATH="/opt/conda/bin:$PATH"
CASO="caso_A"

mkdir -p annotation/${CASO}/integron_results

integron_finder \
  --local-max \
  --func-annot \
  annotation/${CASO}/contigs.fasta \
  --outdir annotation/${CASO}/integron_results

echo "✅ IntegronFinder completo"

In [ ]:
import pathlib, pandas as pd

CASO = "caso_A"
summary_path = list(pathlib.Path(f"annotation/{CASO}/integron_results").glob("*.summary"))[0]
print(summary_path.read_text())

**Preguntas:**

16. ¿Cuántos integrones completos se encontraron?
17. ¿Cuántos elementos In0 y CALIN?
18. ¿En qué contigs se localizan?

## 🔀 Paso 7 — Detección de elementos IS con ISEScan

In [ ]:
%%bash
export PATH="/opt/conda/bin:$PATH"
CASO="caso_A"

isescan.py \
  --seqfile annotation/${CASO}/contigs.fasta \
  --output annotation/${CASO}/isescan_results \
  --nthread 4

echo "✅ ISEScan completo"

In [ ]:
import pandas as pd, pathlib, glob

CASO = "caso_A"
is_files = glob.glob(f"annotation/{CASO}/isescan_results/*.tsv")
if is_files:
    is_df = pd.read_csv(is_files[0], sep="\t")
    print(f"Elementos IS detectados: {len(is_df)}\n")
    print(is_df[["family", "cluster", "seqID", "isBegin", "isEnd"]].to_string())
else:
    print("No se encontraron elementos IS o el archivo aún no está listo.")

**Preguntas:**

19. ¿Cuántos elementos IS se detectaron?
20. ¿En qué contigs se encuentran?
21. ¿Cuáles son las distintas familias de IS identificadas?

## 📊 Paso 8 — Resumen integrado con Python

Construya una tabla resumen que cruce los resultados de Bakta, AMRFinderPlus y PlasmidFinder por contig.

In [ ]:
import pandas as pd

CASO = "caso_A"

# Cargar resultados
amr_df = pd.read_csv(f"annotation/{CASO}/amrfinder_results/amrfinder.tsv", sep="\t")
pf_df  = pd.read_csv(f"annotation/{CASO}/plasmidfinder_results/results_tab.tsv", sep="\t")

# Contigs con genes de resistencia
amr_contigs = set(amr_df["Contig id"].dropna().unique())
# Contigs con plásmidos
plasmid_contigs = set(pf_df["Contig"].dropna().unique())
# Intersección
both = amr_contigs & plasmid_contigs

print("=== Contigs con genes de resistencia ===")
print(sorted(amr_contigs))
print("\n=== Contigs con plásmidos ===")
print(sorted(plasmid_contigs))
print("\n=== Contigs con AMBOS (resistencia + plásmido) ===")
print(sorted(both) if both else "Ninguno")

## 🌿 Paso 9 — (Solo Caso C) Predicción de BGC con antiSMASH

> **Este paso es exclusivo para el Caso C (*Streptomyces venezuelae*)**. Los Casos A y B pueden saltar directamente a las preguntas integradoras.

Los clústeres de genes biosintéticos (BGC) son regiones del genoma que codifican la maquinaria completa para sintetizar un metabolito secundario — antibióticos, antifúngicos, sideróforos, pigmentos, etc. **antiSMASH** es la herramienta estándar para su predicción.

In [ ]:
%%bash
export PATH="/opt/conda/bin:$PATH"
CASO="caso_C"   # ← Solo para Caso C

# Preparar base de datos de antiSMASH
download-antismash-databases --database-dir /opt/antismash_db

echo "✅ Base de datos antiSMASH lista"

In [ ]:
%%bash
export PATH="/opt/conda/bin:$PATH"
CASO="caso_C"

mkdir -p annotation/${CASO}/antismash_results

antismash \
  annotation/${CASO}/contigs.fasta \
  --output-dir annotation/${CASO}/antismash_results \
  --taxon bacteria \
  --genefinding-tool prodigal \
  --cb-knownclusters \
  --cb-subclusterblast \
  --asf \
  --pfam2go \
  --smcog-trees \
  --cpus 4

echo "✅ antiSMASH completo. Reporte en annotation/${CASO}/antismash_results/index.html"

### Explorar los resultados de antiSMASH con Python

In [ ]:
import json, pathlib, pandas as pd

CASO = "caso_C"
json_path = list(pathlib.Path(f"annotation/{CASO}/antismash_results").glob("*.json"))[0]
data = json.loads(json_path.read_text())

# Extraer información de todos los BGC encontrados
registros = []
for record in data.get("records", []):
    for feature in record.get("features", []):
        if feature.get("type") == "region":
            quals = feature.get("qualifiers", {})
            registros.append({
                "Contig": record["id"],
                "BGC_type": ", ".join(quals.get("product", ["desconocido"])),
                "Start": feature["location"]["start"],
                "End": feature["location"]["end"],
                "Longitud_pb": feature["location"]["end"] - feature["location"]["start"],
                "KnownCluster_hit": quals.get("knownclusterblast", ["—"])[0][:60] if quals.get("knownclusterblast") else "—"
            })

bgc_df = pd.DataFrame(registros)
print(f"Total de BGC predichos: {len(bgc_df)}\n")
print(bgc_df.to_string())

In [ ]:
# Distribución de tipos de BGC
import matplotlib.pyplot as plt

tipo_counts = bgc_df["BGC_type"].value_counts()
fig, ax = plt.subplots(figsize=(8, 4))
tipo_counts.plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("Número de BGC")
ax.set_title(f"Tipos de BGC predichos en S. venezuelae ATCC 10712\n(antiSMASH, n={len(bgc_df)})")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(f"annotation/{CASO}/antismash_results/bgc_tipos.png", dpi=150)
plt.show()
print("✅ Gráfico guardado")

**Preguntas — Caso C (antiSMASH):**

26. ¿Cuántos BGC predijo antiSMASH en el genoma de *S. venezuelae*?
27. ¿Qué tipos de BGC están presentes? ¿Cuál es el más frecuente?
28. ¿Cuál BGC corresponde al clúster del **cloranfenicol** según KnownClusterBlast?
29. ¿Hay BGC sin homología conocida en MIBiG? ¿Cuántos? ¿Qué implicaría caracterizar uno de ellos?
30. Elija un BGC que le llame la atención y describa: tipo, tamaño en pb y metabolito predicho.

## ❓ Preguntas integradoras

Responda estas preguntas en una celda de texto nueva (`+ Texto`) al final del notebook:

**Casos A y B:**

31. Considerando los resultados de Bakta, AMRFinderPlus y PlasmidFinder: ¿qué puede concluir sobre la movilidad de los genes de resistencia?
32. Si un gen de resistencia está en un plásmido que también contiene un integrón, ¿qué implica para la diseminación de resistencias?
33. ¿Todos los contigs con plásmidos pertenecen claramente al organismo de estudio? ¿Cómo lo verificaría?
34. Para el **Caso B** (*K. pneumoniae*): ¿los resultados de AMRFinderPlus son consistentes con el perfil de resistencia a carbapenemes descrito en el artículo?

**Caso C:**

35. ¿Cuántos BGC identificó Bakta en su anotación estándar? ¿Coincide con los predichos por antiSMASH? ¿Por qué podrían diferir?
36. Compare el número de BGC predichos con lo reportado en la literatura para *S. venezuelae* ATCC 10712. ¿Cuántos tienen producto conocido y cuántos son "silenciosos" o sin homología en MIBiG?
37. ¿Qué ventaja tiene usar un genoma *finished* (como GCF_000253235.1) en lugar de un borrador (*draft*) con muchos contigs para este tipo de análisis?

---
*Consulte la [guía compartida](00_genome_annotation_common.md) y el [README del Módulo 6](../README.md) para repasar los conceptos.*
